<a href="https://colab.research.google.com/github/Priyall33/Pcos-Endometrosis-risk-model/blob/main/03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 — Feature Engineering

This notebook takes the raw datasets from Notebooks 01 and 02 and prepares them for machine learning modeling.

Feature engineering is the bridge between raw data and a working model. Without this step the model would receive messy, unscaled, or improperly formatted data and produce unreliable results.

Preparing NHANES and BRFSS datasets for model

In [10]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Mount Google Drive — where our datasets are saved
drive.mount('/content/drive')
os.makedirs('outputs', exist_ok=True)
os.makedirs('/content/drive/MyDrive/model_data', exist_ok=True)

print("Save path: /content/drive/MyDrive/model_data/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Save path: /content/drive/MyDrive/model_data/


# NHANES

In [13]:
# Load from Google Drive
df_nhanes = pd.read_csv('/content/drive/MyDrive/Pcos Endo Project /nhanes_stacked_all_cycles.csv')

print("NHANES DATASET")
print(f"Shape: {df_nhanes.shape}")
print(f"\nColumns: {list(df_nhanes.columns)}")
print(f"\nEndometriosis label distribution:")
print(df_nhanes['endometriosis'].value_counts())
print(f"\nMissing data:")
for col in df_nhanes.columns:
    pct = df_nhanes[col].isna().mean() * 100
    if pct > 0:
        print(f"  {col}: {pct:.1f}% missing")

NHANES DATASET
Shape: (11344, 16)

Columns: ['SEQN', 'LBXAMH', 'LBXFSH', 'LBXLUH', 'LBXSHBG', 'LBXAND', 'LBXEST', 'LBXPG4', 'RHQ305', 'RHQ010', 'RHD043', 'RHQ131', 'RHQ160', 'RHQ162', 'cycle', 'endometriosis']

Endometriosis label distribution:
endometriosis
0    10016
1     1328
Name: count, dtype: int64

Missing data:
  LBXAMH: 70.5% missing
  LBXFSH: 70.5% missing
  LBXLUH: 70.5% missing
  LBXSHBG: 30.9% missing
  LBXAND: 70.5% missing
  LBXEST: 30.1% missing
  LBXPG4: 70.7% missing
  RHQ010: 0.0% missing
  RHD043: 56.5% missing
  RHQ131: 0.1% missing
  RHQ160: 16.6% missing
  RHQ162: 16.6% missing


In [15]:
# Dropped:
# SEQN
# RHQ305
# RHD043
# RHQ162

nhanes_features = [
    # Hormone biomarkers
    'LBXAMH',   # AMH — strongest single PCOS biomarker
    'LBXFSH',   # FSH — ovarian function indicator
    'LBXLUH',   # LH — elevated LH:FSH ratio is PCOS hallmark
    'LBXSHBG',  # SHBG — low = more free testosterone
    'LBXAND',   # Androstenedione — hyperandrogenism marker
    'LBXEST',   # Estradiol — endometriosis marker
    'LBXPG4',   # Progesterone — menstrual cycle phase

    # Reproductive health features
    'RHQ010',   # Age at first period — early menarche = endo risk
    'RHQ131',   # Ever pregnant — affects hormone profile
    'RHQ160',   # Ever had fibroids — co-occurs with endometriosis

    # Survey metadata
    'cycle',    # Which NHANES cycle
]

# Target
nhanes_target = 'endometriosis'


available = [f for f in nhanes_features if f in df_nhanes.columns]
missing = [f for f in nhanes_features if f not in df_nhanes.columns]

print(" NHANES FEATURE SELECTION ")
print(f"Features selected : {len(available)}")
print(f"Target            : {nhanes_target}")
if missing:
    print(f"Not found         : {missing}")

print(f"\nFinal feature list:")
for f in available:
    pct_missing = df_nhanes[f].isna().mean() * 100
    print(f"  {f:<12} — {pct_missing:.1f}% missing")

# X and y
X_nhanes = df_nhanes[available].copy()
y_nhanes = df_nhanes[nhanes_target].copy()

print(f"\nX shape: {X_nhanes.shape}")
print(f"y shape: {y_nhanes.shape}")
print(f"Positive cases: {y_nhanes.sum():,} ({y_nhanes.mean()*100:.1f}%)")

 NHANES FEATURE SELECTION 
Features selected : 11
Target            : endometriosis

Final feature list:
  LBXAMH       — 70.5% missing
  LBXFSH       — 70.5% missing
  LBXLUH       — 70.5% missing
  LBXSHBG      — 30.9% missing
  LBXAND       — 70.5% missing
  LBXEST       — 30.1% missing
  LBXPG4       — 70.7% missing
  RHQ010       — 0.0% missing
  RHQ131       — 0.1% missing
  RHQ160       — 16.6% missing
  cycle        — 0.0% missing

X shape: (11344, 11)
y shape: (11344,)
Positive cases: 1,328 (11.7%)


Categorial Variables

In [16]:
X_nhanes = X_nhanes.copy()

#Ever been pregnant
if 'RHQ131' in X_nhanes.columns:
    X_nhanes['RHQ131'] = X_nhanes['RHQ131'].map({1.0: 1, 2.0: 0})
    print(f"RHQ131 (ever pregnant)  : recoded 2→0")
    print(f"  Values: {X_nhanes['RHQ131'].value_counts().to_dict()}")

#Ever told had fibroids
if 'RHQ160' in X_nhanes.columns:
    X_nhanes['RHQ160'] = X_nhanes['RHQ160'].map({1.0: 1, 2.0: 0})
    print(f"\nRHQ160 (ever had fibroids): recoded 2→0")
    print(f"  Values: {X_nhanes['RHQ160'].value_counts().to_dict()}")

#convert text cycle names to numeric codes
if 'cycle' in X_nhanes.columns:
    cycle_map = {
        '2011-2012': 0,
        '2013-2014': 1,
        '2015-2016': 2,
        '2017-2020': 3
    }
    X_nhanes['cycle'] = X_nhanes['cycle'].map(cycle_map)
    print(f"\ncycle: converted to numeric codes")
    print(f"  Values: {X_nhanes['cycle'].value_counts().to_dict()}")

print(f"\nFinal X_nhanes sample:")
print(X_nhanes.head(3))

RHQ131 (ever pregnant)  : recoded 2→0
  Values: {1.0: 9459, 0.0: 1868}

RHQ160 (ever had fibroids): recoded 2→0
  Values: {0.0: 2247, 1.0: 1275}

cycle: converted to numeric codes
  Values: {3: 3985, 1: 2601, 2: 2505, 0: 2253}

✅ Encoding complete

Final X_nhanes sample:
   LBXAMH  LBXFSH  LBXLUH  LBXSHBG  LBXAND  LBXEST   LBXPG4  RHQ010  RHQ131  \
0    1.65    7.15    4.55    22.54    93.1   35.50     1.50    13.0     0.0   
1    0.40    2.81    3.82    48.18    86.0  142.00  1540.00     9.0     1.0   
2    0.02   33.10   20.48    30.37    51.6    1.22     1.36    13.0     1.0   

   RHQ160  cycle  
0     NaN      3  
1     NaN      3  
2     NaN      3  


Train/Test Split

In [19]:
# SMOTE will be applied ONLY to the training set

from sklearn.model_selection import train_test_split

X_train_nhanes, X_test_nhanes, y_train_nhanes, y_test_nhanes = train_test_split(
    X_nhanes,
    y_nhanes,
    test_size=0.2,
    random_state=42,
    stratify=y_nhanes
)

print("NHANES TRAIN/TEST SPLIT ")
print(f"Total rows     : {len(X_nhanes):,}")
print(f"Training set   : {X_train_nhanes.shape[0]:,} rows ({X_train_nhanes.shape[0]/len(X_nhanes)*100:.0f}%)")
print(f"Test set       : {X_test_nhanes.shape[0]:,} rows ({X_test_nhanes.shape[0]/len(X_nhanes)*100:.0f}%)")

print(f"\nTraining label distribution:")
print(y_train_nhanes.value_counts())
print(f"Positive rate  : {y_train_nhanes.mean()*100:.1f}%")

print(f"\nTest label distribution:")
print(y_test_nhanes.value_counts())
print(f"Positive rate  : {y_test_nhanes.mean()*100:.1f}%")

NHANES TRAIN/TEST SPLIT 
Total rows     : 11,344
Training set   : 9,075 rows (80%)
Test set       : 2,269 rows (20%)

Training label distribution:
endometriosis
0    8013
1    1062
Name: count, dtype: int64
Positive rate  : 11.7%

Test label distribution:
endometriosis
0    2003
1     266
Name: count, dtype: int64
Positive rate  : 11.7%


# BRFSS

In [31]:
df_brfss = pd.read_csv('/content/drive/MyDrive/Pcos Endo Project /brfss_model2_final.csv')

print(" BRFSS DATASET ")
print(f"Shape: {df_brfss.shape}")
print(f"\nColumns: {list(df_brfss.columns)}")
print(f"\nProxy label distribution:")
print(df_brfss['high_risk'].value_counts())
print(f"High risk rate: {df_brfss['high_risk'].mean()*100:.1f}%")
print(f"\nMissing data per column:")
for col in df_brfss.columns:
    pct = df_brfss[col].isna().mean() * 100
    flag = "⚠️" if pct > 20 else "✅"
    print(f"  {flag} {col}: {pct:.1f}% missing")

 BRFSS DATASET 
Shape: (229541, 15)

Columns: ['BMI', 'insulin_resistance', 'takes_insulin', 'prediabetes', 'has_diabetes', 'BPHIGH6', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'ADDEPEV3', 'EXERANY2', 'INCOME3', '_AGE80', '_RACE', 'high_risk']

Proxy label distribution:
high_risk
1    134365
0     95176
Name: count, dtype: int64
High risk rate: 58.5%

Missing data per column:
  ✅ BMI: 12.3% missing
  ✅ insulin_resistance: 0.0% missing
  ✅ takes_insulin: 0.0% missing
  ✅ prediabetes: 0.0% missing
  ✅ has_diabetes: 0.0% missing
  ✅ BPHIGH6: 0.4% missing
  ✅ GENHLTH: 0.2% missing
  ✅ PHYSHLTH: 5.0% missing
  ✅ MENTHLTH: 4.0% missing
  ✅ ADDEPEV3: 0.5% missing
  ✅ EXERANY2: 0.3% missing
  ⚠️ INCOME3: 45.6% missing
  ✅ _AGE80: 1.6% missing
  ✅ _RACE: 4.1% missing
  ✅ high_risk: 0.0% missing


In [32]:
df_brfss = pd.read_csv('/content/drive/MyDrive/brfss_model2_final.csv')

print("Available columns:", list(df_brfss.columns))
print(f"Shape: {df_brfss.shape}")

# Rebuild risk score from component flags
risk_cols = [
    'high_bmi',
    'insulin_resistance',
    'poor_physical_health',
    'poor_mental_health',
    'no_insurance',
    'inactive',
    'depression'
]

available_risk = [c for c in risk_cols if c in df_brfss.columns]
print(f"\nRisk columns found: {available_risk}")

risk_score = df_brfss[available_risk].sum(axis=1)

# Test thresholds
print("\n THRESHOLD ANALYSIS ")
for threshold in [2, 3, 4, 5]:
    pct = (risk_score >= threshold).mean() * 100
    count = (risk_score >= threshold).sum()
    print(f"  Threshold {threshold}+: {count:,} high risk ({pct:.1f}%)")

print(f"\nTarget: 10-15% to match real world PCOS/endo prevalence")

Available columns: ['BMI', 'PHYSHLTH', 'MENTHLTH', '_HLTHPL1', 'EXERANY2', 'ADDEPEV3', 'BPHIGH6', 'GENHLTH', 'insulin_resistance', 'takes_insulin', 'prediabetes', 'has_diabetes', 'high_bmi', 'poor_physical_health', 'poor_mental_health', 'no_insurance', 'inactive', 'depression', '_AGE80', '_RACE', 'INCOME3', 'high_risk']
Shape: (229541, 22)

Risk columns found: ['high_bmi', 'insulin_resistance', 'poor_physical_health', 'poor_mental_health', 'no_insurance', 'inactive', 'depression']

=== THRESHOLD ANALYSIS ===
  Threshold 2+: 200,542 high risk (87.4%)
  Threshold 3+: 134,365 high risk (58.5%)
  Threshold 4+: 62,818 high risk (27.4%)
  Threshold 5+: 20,080 high risk (8.7%)

Target: 10-15% to match real world PCOS/endo prevalence


In [34]:
# Rebuild risk score
risk_cols = [
    'high_bmi', 'insulin_resistance', 'poor_physical_health',
    'poor_mental_health', 'no_insurance', 'inactive', 'depression'
]

risk_score = df_brfss[risk_cols].sum(axis=1)

# Apply threshold of 5
df_brfss['high_risk'] = (risk_score >= 5).astype(int)

print("CORRECTED PROXY LABEL ")
print(f"Total women    : {len(df_brfss):,}")
print(f"High risk (1)  : {df_brfss['high_risk'].sum():,} ({df_brfss['high_risk'].mean()*100:.1f}%)")
print(f"Low risk  (0)  : {(df_brfss['high_risk']==0).sum():,} ({(df_brfss['high_risk']==0).mean()*100:.1f}%)")
print(f"\nThis is much closer to real world PCOS/endo")
print(f"prevalence of 10-15%")

CORRECTED PROXY LABEL 
Total women    : 229,541
High risk (1)  : 20,080 (8.7%)
Low risk  (0)  : 209,461 (91.3%)

This is much closer to real world PCOS/endo
prevalence of 10-15%


In [35]:
# DROPPED:
# INCOME3
# high_bmi
# poor_physical_health
# poor_mental_health
# no_insurance
# inactive
# depression

brfss_features = [
    # Metabolic markers
    'BMI',               # continuous BMI value
    'insulin_resistance',# combined IR flag (prediabetes/diabetes/insulin)
    'takes_insulin',     # currently taking insulin
    'prediabetes',       # told have prediabetes
    'has_diabetes',      # diabetes diagnosis
    'BPHIGH6',           # high blood pressure

    # Health status — raw values not binary flags
    'GENHLTH',           # general health 1-5 scale
    'PHYSHLTH',          # days physical health not good (0-30)
    'MENTHLTH',          # days mental health not good (0-30)
    'ADDEPEV3',          # depression diagnosis

    # Lifestyle
    'EXERANY2',          # physical activity

    # Demographics
    '_AGE80',            # age
    '_RACE',             # race/ethnicity
]

# Target
brfss_target = 'high_risk'

# Validate
available_brfss = [f for f in brfss_features if f in df_brfss.columns]
missing_brfss = [f for f in brfss_features if f not in df_brfss.columns]

print("BRFSS FEATURE SELECTION ")
print(f"Features selected : {len(available_brfss)}")
if missing_brfss:
    print(f"Not found         : {missing_brfss}")

print(f"\nFinal feature list:")
for f in available_brfss:
    pct = df_brfss[f].isna().mean() * 100
    print(f"  {f:<20} — {pct:.1f}% missing")

#  X and y
X_brfss = df_brfss[available_brfss].copy()
y_brfss = df_brfss[brfss_target].copy()

# Drop rows where target is missing
mask = y_brfss.notna()
X_brfss = X_brfss[mask]
y_brfss = y_brfss[mask]

print(f"\nX shape : {X_brfss.shape}")
print(f"y shape : {y_brfss.shape}")
print(f"High risk cases : {y_brfss.sum():,} ({y_brfss.mean()*100:.1f}%)")

=== BRFSS FEATURE SELECTION ===
Features selected : 13

Final feature list:
  BMI                  — 12.3% missing
  insulin_resistance   — 0.0% missing
  takes_insulin        — 0.0% missing
  prediabetes          — 0.0% missing
  has_diabetes         — 0.0% missing
  BPHIGH6              — 0.4% missing
  GENHLTH              — 0.2% missing
  PHYSHLTH             — 5.0% missing
  MENTHLTH             — 4.0% missing
  ADDEPEV3             — 0.5% missing
  EXERANY2             — 0.3% missing
  _AGE80               — 1.6% missing
  _RACE                — 4.1% missing

X shape : (229541, 13)
y shape : (229541,)
High risk cases : 20,080 (8.7%)


Train/Test Split

In [37]:
from sklearn.model_selection import train_test_split

X_train_brfss, X_test_brfss, y_train_brfss, y_test_brfss = train_test_split(
    X_brfss,
    y_brfss,
    test_size=0.2,
    random_state=42,
    stratify=y_brfss
)

print("BRFSS TRAIN/TEST SPLIT ")
print(f"Total rows     : {len(X_brfss):,}")
print(f"Training set   : {X_train_brfss.shape[0]:,} rows ({X_train_brfss.shape[0]/len(X_brfss)*100:.0f}%)")
print(f"Test set       : {X_test_brfss.shape[0]:,} rows ({X_test_brfss.shape[0]/len(X_brfss)*100:.0f}%)")
print(f"\nTraining label distribution:")
print(y_train_brfss.value_counts())
print(f"High risk rate : {y_train_brfss.mean()*100:.1f}%")
print(f"\nTest label distribution:")
print(y_test_brfss.value_counts())
print(f"High risk rate : {y_test_brfss.mean()*100:.1f}%")
print(f"\n✅ Stratification maintained in both splits")

BRFSS TRAIN/TEST SPLIT 
Total rows     : 229,541
Training set   : 183,632 rows (80%)
Test set       : 45,909 rows (20%)

Training label distribution:
high_risk
0    167568
1     16064
Name: count, dtype: int64
High risk rate : 8.7%

Test label distribution:
high_risk
0    41893
1     4016
Name: count, dtype: int64
High risk rate : 8.7%

✅ Stratification maintained in both splits


Saving all files

In [41]:
# saving X and y separately because SMOTE needs them separately to oversample only the training features and labels

import os
save_dir = '/content/drive/MyDrive/model_data/'
os.makedirs(save_dir, exist_ok=True)

# Save NHANES splits
X_train_nhanes.to_csv(f'{save_dir}nhanes_X_train.csv', index=False)
X_test_nhanes.to_csv(f'{save_dir}nhanes_X_test.csv', index=False)
y_train_nhanes.to_csv(f'{save_dir}nhanes_y_train.csv', index=False)
y_test_nhanes.to_csv(f'{save_dir}nhanes_y_test.csv', index=False)

# Save BRFSS splits
X_train_brfss.to_csv(f'{save_dir}brfss_X_train.csv', index=False)
X_test_brfss.to_csv(f'{save_dir}brfss_X_test.csv', index=False)
y_train_brfss.to_csv(f'{save_dir}brfss_y_train.csv', index=False)
y_test_brfss.to_csv(f'{save_dir}brfss_y_test.csv', index=False)

print(" All files saved to Google Drive!")
print(f"\nSUMMARY ")
print(f"NHANES Model 1:")
print(f"  Training : {X_train_nhanes.shape[0]:,} rows × {X_train_nhanes.shape[1]} features")
print(f"  Test     : {X_test_nhanes.shape[0]:,} rows × {X_test_nhanes.shape[1]} features")
print(f"  Positive rate: 11.7% (pre-SMOTE)")
print(f"\nBRFSS Model 2:")
print(f"  Training : {X_train_brfss.shape[0]:,} rows × {X_train_brfss.shape[1]} features")
print(f"  Test     : {X_test_brfss.shape[0]:,} rows × {X_test_brfss.shape[1]} features")
print(f"  Positive rate: 8.7% (pre-SMOTE)")
print(f"\nFiles saved:")
for f in ['nhanes_X_train', 'nhanes_X_test', 'nhanes_y_train', 'nhanes_y_test',
          'brfss_X_train', 'brfss_X_test', 'brfss_y_train', 'brfss_y_test']:
    print(f"  {f}.csv")


 All files saved to Google Drive!

SUMMARY 
NHANES Model 1:
  Training : 9,075 rows × 11 features
  Test     : 2,269 rows × 11 features
  Positive rate: 11.7% (pre-SMOTE)

BRFSS Model 2:
  Training : 183,632 rows × 13 features
  Test     : 45,909 rows × 13 features
  Positive rate: 8.7% (pre-SMOTE)

Files saved:
  nhanes_X_train.csv
  nhanes_X_test.csv
  nhanes_y_train.csv
  nhanes_y_test.csv
  brfss_X_train.csv
  brfss_X_test.csv
  brfss_y_train.csv
  brfss_y_test.csv
